In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("iot_sensor_data_raw.csv")

In [3]:
df.head()

,Timestamp,Device_ID,Temperature,Humidity,Pressure,Vibration,Battery_Level,Location,Machine_Status
0,2026-04-21 22:30:00,DEV_005,65.89,62.70,1007.96,2.22,46.51,Factory_A,Warning
1,2026-01-22 06:15:00,DEV_007,70.65,70.95,1021.85,2.73,57.04,Factory_A,Normal
2,2026-04-01 07:00:00,DEV_005,81.01,60.49,1009.82,2.15,11.65,Factory_A,Normal
3,2026-01-12 14:30:00,DEV_007,74.17,68.76,1001.42,2.03,17.36,Factory_C,Critical
4,2026-05-25 19:30:00,DEV_008,74.04,54.32,1014.34,NaN,27.04,Factory_C,Normal


In [4]:
df.shape

(20000, 9)

In [5]:
df.describe()

,Temperature,Humidity,Pressure,Vibration,Battery_Level
count,19701.000000,19700.000000,19700.000000,19702.000000,20000.000000
mean,70.218406,55.001890,1012.890236,2.534875,54.528231
std,8.684540,12.061963,14.999115,1.015491,26.048708
min,34.170000,8.900000,956.340000,-0.480000,2.011730
25%,64.720000,46.930000,1002.720000,1.960000,31.997500
50%,70.040000,54.990000,1012.790000,2.500000,54.335000
75%,75.410000,63.170000,1022.980000,3.050000,76.930000
max,159.746758,98.910000,1070.900000,19.993705,99.990000


In [8]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
print(df['Timestamp'].dtypes)

datetime64[ns]


In [9]:

is_ordered = df['Timestamp'].is_monotonic_increasing

if is_ordered:
    print("✅ The timestamps are correctly ordered chronologically.")
else:
    print("❌ The timestamps are NOT ordered correctly.")


❌ The timestamps are NOT ordered correctly.


In [10]:
columnsContaingMissingValues = df.columns[df.isna().any()].tolist()

In [11]:
columnsContaingMissingValues

['Temperature', 'Humidity', 'Pressure', 'Vibration']

In [19]:

missing_rows = df[df[columnsContaingMissingValues].isna()]

print(f"Total missing sensor readings: {len(missing_rows)}")


Total missing sensor readings: 20000


In [21]:
missing_percentage = df.isna().mean() * 100

missing_df = missing_percentage.to_frame(name='Missing %').sort_values(by='Missing %', ascending=False)
print(missing_df)


                Missing %
Pressure            1.500
Humidity            1.500
Temperature         1.495
Vibration           1.490
Timestamp           0.000
Device_ID           0.000
Battery_Level       0.000
Location            0.000
Machine_Status      0.000


In [22]:
df[columnsContaingMissingValues] = df[columnsContaingMissingValues].fillna(df[columnsContaingMissingValues].median())

In [23]:
missing_percentage = df.isna().mean() * 100

missing_df = missing_percentage.to_frame(name='Missing %').sort_values(by='Missing %', ascending=False)
print(missing_df)


                Missing %
Timestamp             0.0
Device_ID             0.0
Temperature           0.0
Humidity              0.0
Pressure              0.0
Vibration             0.0
Battery_Level         0.0
Location              0.0
Machine_Status        0.0


In [38]:
def abnormalValues(columnName):
    Q1 = df[columnName].quantile(0.25)
    Q3 = df[columnName].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    abnormal_temp = df[(df[columnName] < lower_bound) | (df[columnName] > upper_bound)]
    
    print(f"Acceptable range: {lower_bound:.2f} to {upper_bound:.2f}")
    print(f"Total abnormal readings detected: {len(abnormal_temp)}")
    print(abnormal_temp[['Timestamp', columnName]].head())


In [39]:
abnormalValues('Temperature')

Acceptable range: 49.09 to 91.06
Total abnormal readings detected: 203
              Timestamp  Temperature
77  2026-01-11 05:00:00   150.182051
337 2026-07-11 07:30:00   133.908634
488 2026-03-16 14:15:00    93.350000
596 2026-06-20 00:15:00   152.454296
724 2026-07-17 01:30:00   150.099637


In [40]:
abnormalValues('Vibration')

Acceptable range: 0.36 to 4.64
Total abnormal readings detected: 185
              Timestamp  Vibration
204 2026-06-14 03:15:00       4.71
341 2026-05-01 16:00:00       0.26
471 2026-02-09 12:45:00       5.30
758 2026-01-03 04:30:00      -0.10
759 2026-07-09 23:30:00       4.77


In [35]:
# 1. Define your critical battery threshold (e.g., 15%)
CRITICAL_THRESHOLD = 15

# 2. Filter rows where battery levels are critically low
low_battery_df = df[df['Battery_Level'] <= CRITICAL_THRESHOLD]

# 3. Sort by lowest battery first for prioritization
low_battery_df = low_battery_df.sort_values(by='Battery_Level', ascending=True)

print(f"Total machines requiring immediate battery replacement: {low_battery_df['Device_ID'].nunique()}")
print(low_battery_df[['Device_ID', 'Battery_Level']].head())


Total machines requiring immediate battery replacement: 8
      Device_ID  Battery_Level
4861    DEV_008       2.011730
2328    DEV_007       2.146086
8158    DEV_002       2.154615
18694   DEV_006       2.201033
9727    DEV_001       2.257059


In [42]:
def calculateAverage(columnName,groupByField):
    avg_temp_by_hour = df.groupby(groupByField)[columnName].mean()
    
    print("Average columnName:")
    print(avg_temp_by_hour)


In [43]:
calculateAverage('Temperature',df['Timestamp'].dt.hour)

Average columnName:
Timestamp
0     70.083080
1     70.543074
2     69.924102
3     69.717616
4     69.808481
5     70.263542
6     70.250108
7     70.647096
8     70.389791
9     70.972674
10    69.931012
11    69.892213
12    69.992680
13    70.126029
14    69.838031
15    70.221210
16    69.966825
17    70.544828
18    70.221214
19    69.997957
20    70.134827
21    70.139495
22    71.171403
23    70.402801
Name: Temperature, dtype: float64


In [44]:
calculateAverage('Temperature','Device_ID')

Average columnName:
Device_ID
DEV_001    70.248192
DEV_002    70.370555
DEV_003    69.935013
DEV_004    70.576572
DEV_005    70.116720
DEV_006    70.006095
DEV_007    70.145846
DEV_008    70.315545
Name: Temperature, dtype: float64


In [45]:
calculateAverage('Vibration','Device_ID')

Average columnName:
Device_ID
DEV_001    2.552866
DEV_002    2.563576
DEV_003    2.538532
DEV_004    2.524378
DEV_005    2.494359
DEV_006    2.553233
DEV_007    2.520977
DEV_008    2.527996
Name: Vibration, dtype: float64


In [46]:
max_temp_by_device = df.groupby('Device_ID')['Temperature'].max().sort_values(ascending=False)
print(max_temp_by_device)


Device_ID
DEV_001    159.746758
DEV_003    157.603693
DEV_007    153.877632
DEV_008    153.622123
DEV_005    152.974450
DEV_004    151.965273
DEV_002    151.818550
DEV_006    136.003203
Name: Temperature, dtype: float64


In [47]:
min_battery_by_device = df.groupby('Device_ID')['Battery_Level'].min().sort_values()
print(min_battery_by_device)


Device_ID
DEV_008    2.011730
DEV_007    2.146086
DEV_002    2.154615
DEV_006    2.201033
DEV_001    2.257059
DEV_005    2.956738
DEV_003    3.670604
DEV_004    8.514009
Name: Battery_Level, dtype: float64


In [48]:
# Computes the means, finds the maximum value's index label
highest_vib_device = df.groupby('Device_ID')['Vibration'].mean().idxmax()
highest_vib_val = df.groupby('Device_ID')['Vibration'].mean().max()

print(f"Device with highest average vibration: {highest_vib_device} ({highest_vib_val:.2f})")


Device with highest average vibration: DEV_002 (2.56)


In [52]:
# Groups by factory site, finds the maximum temperature index label
hottest_factory = df.groupby('Location')['Temperature'].mean().idxmax()
hottest_factory_val = df.groupby('Location')['Temperature'].mean().max()

print(f"Factory with highest average temperature: {hottest_factory} ({hottest_factory_val:.2f})")


Factory with highest average temperature: Factory_C (70.27)
